In [63]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import scraping_functions as sf
import importlib
import time
import re

importlib.reload(sf);

In [46]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [ ]:
def get_page(url, headers, retries=5):
    """
    Sends an HTTP GET request to a given URL and retries the request
    if the page is not successfully retrieved.

    The function attempts to access the requested page up to the specified
    number of retries. If a request returns HTTP status code 200, the
    response is immediately returned. Otherwise, the function waits
    three seconds before trying again.

    Parameters:
        url (str): URL of the page to be requested.
        headers (dict): HTTP headers used when sending the request.
        retries (int): Maximum number of request attempts. Defaults to 5.

    Returns:
        requests.Response: The successful response, or the response from
        the final attempt if all retries fail.
    """
    for attempt in range(retries):
        response = requests.get(url, headers=headers)

        if response.status_code == 200:
            return response

        time.sleep(3)
    # If every attempt fails, return the response from the final request
    return response

In [4]:
url = f'https://www.transfermarkt.com/premier-league/spieltag/wettbewerb/GB1/saison_id/2025/spieltag/1'
response = get_page(url, headers)
soup = BeautifulSoup(response.content, "lxml")

## Matches csv

In [5]:
all_matches = soup.find_all('table',{'style':'border-top: 0 !important;'})

home_team = 'hauptlink zentriert no-border-links no-border-rechts hide-for-small spieltagsansicht-wappen'
away_team = 'hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen'

# SCRAPING AWAY INFORMATION
print('-----------------------------------------')
print('SCRAPING AWAY INFORMATION')
print('-----------------------------------------')

away_team_info = all_matches[5].find_all('td',{'class':away_team})

away_team_url = away_team_info[0].find('a').get('href')
away_team_id = away_team_url.split('/')[-3]
away_team_name = away_team_info[0].find('a').get('title')

display(away_team_info,away_team_url,away_team_id,away_team_name)

# SCRAPING HOME INFORMATION
print('-----------------------------------------')
print('SCRAPING HOME INFORMATION')
print('-----------------------------------------')

home_team_info = all_matches[5].find_all('td',{'class':home_team})

home_team_url = home_team_info[0].find('a').get('href')
home_team_id = home_team_url.split('/')[-3]
home_team_name = home_team_info[0].find('a').get('title')

display(home_team_info,home_team_url,home_team_id,home_team_name)

-----------------------------------------
SCRAPING AWAY INFORMATION
-----------------------------------------


[<td class="hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen">
 <a href="/manchester-city/spielplan/verein/281/saison_id/2025" title="Manchester City"><img alt="Manchester City" class="" src="https://img.a.transfermarkt.technology/wappen/small/281.png?lm=4711" title="Manchester City"/></a> </td>]

'/manchester-city/spielplan/verein/281/saison_id/2025'

'281'

'Manchester City'

-----------------------------------------
SCRAPING HOME INFORMATION
-----------------------------------------


[<td class="hauptlink zentriert no-border-links no-border-rechts hide-for-small spieltagsansicht-wappen">
 <a href="/wolverhampton-wanderers/spielplan/verein/543/saison_id/2025" title="Wolverhampton Wanderers"><img alt="Wolverhampton Wanderers" class="" src="https://img.a.transfermarkt.technology/wappen/small/543.png?lm=4711" title="Wolverhampton Wanderers"/></a> </td>]

'/wolverhampton-wanderers/spielplan/verein/543/saison_id/2025'

'543'

'Wolverhampton Wanderers'

In [6]:
match_url = all_matches[0].find('td',{'class':'spieltagsansicht-ergebnis'}).find('a').get('href')

match_id = match_url.split('/')[-1]

match_result = all_matches[0].find('span',{'class':'matchresult finished'}).string

match_info = all_matches[0].find_all('td',{'class':'zentriert no-border'})

match_day = match_info[0].find('a').get('href').split('/')[-1]
match_referee = match_info[1].find('a').string
match_attendance = match_info[2].get_text().strip()
match_attendance = re.sub('[.]','', match_attendance).split(' ')[0]
time_info = match_info[0].find('a').next_sibling.strip().removeprefix('-').strip().split(' ')
match_time = time_info[0]
match_time_period = time_info[-1]

display(match_info,match_day,match_referee,match_attendance,match_time,match_time_period,match_result,match_url,match_id)


[<td class="zentriert no-border" colspan="5">
 <div class="di"><span class="hide-for-small">Friday,</span><span class="show-for-small">Fri</span></div> <a href="/aktuell/waspassiertheute/aktuell/new/datum/2025-08-15">
                                                                 15/08/2025                                        </a>
                                                              - 9:00 PM
                             </td>,
 <td class="zentriert no-border" colspan="5">
 <span>Referee: <a href="/anthony-taylor/profil/schiedsrichter/847" title="Anthony Taylor">Anthony Taylor</a></span> </td>,
 <td class="zentriert no-border" colspan="5">
 <span class="icons_sprite icon-zuschauer-zahl" title="Attendance"> </span>
                                         60.315                                </td>]

'2025-08-15'

'Anthony Taylor'

'60315'

'9:00'

'PM'

'4:2'

'/spielbericht/index/spielbericht/4625774'

'4625774'

In [7]:
display(home_team_url,home_team_id,home_team_name,match_result,away_team_url,away_team_id,away_team_name,match_day,match_referee,match_attendance,match_time,match_time_period)

'/wolverhampton-wanderers/spielplan/verein/543/saison_id/2025'

'543'

'Wolverhampton Wanderers'

'4:2'

'/manchester-city/spielplan/verein/281/saison_id/2025'

...

'2025-08-15'

'Anthony Taylor'

'60315'

'9:00'

'PM'

## Events csv

In [8]:
match_event = all_matches[0].find_all('tr',{'class':'no-border spieltagsansicht-aktionen'})

display(match_event)

[<tr class="no-border spieltagsansicht-aktionen">
 <td class="rechts no-border-rechts spieltagsansicht"><div class="di"><div class="di nowrap"><span class="hide-for-small"><a href="/hugo-ekitike/profil/spieler/709726" title="Hugo Ekitiké">Hugo Ekitiké</a></span></div><div class="di nowrap"><span class="show-for-small"><a href="/hugo-ekitike/profil/spieler/709726" title="Hugo Ekitiké">H. Ekitiké</a></span></div></div><span class="icons_sprite icon-tor-formation" title="Minute 37: Goal"> </span></td>
 <td class="zentriert no-border-links">37'</td>
 <td class="zentriert hauptlink">1:0</td>
 <td class="zentriert no-border-rechts"> </td>
 <td class="links no-border-links"> </td>
 </tr>,
 <tr class="no-border spieltagsansicht-aktionen">
 <td class="rechts no-border-rechts spieltagsansicht"><div class="di"><div class="di nowrap"><span class="hide-for-small"><a href="/cody-gakpo/profil/spieler/434675" title="Cody Gakpo">Cody Gakpo</a></span></div><div class="di nowrap"><span class="show-for-sm

In [9]:
player_url = match_event[0].find('td',{'class':'spieltagsansicht'}).find('a').get('href')
player_id = player_url.split('/')[-1]
player_name = match_event[0].find('td',{'class':'spieltagsansicht'}).find('a').get('title')

event_type = match_event[0].find('span',{'class':'icons_sprite'}).get('class')[-1]

check = match_event[0].find('td',{'class':'zentriert hauptlink'})
event_score = None if check == None else check.string

home='links'
away='rechts'

check = lambda x: match_event[0].find('td',{'class':f'zentriert no-border-{x}'}).string
event_time_label = check(away) if check(home) == '\xa0' else check(home)

time_list = re.sub("[']",'', event_time_label).split('+')
event_time_minute = int(time_list[0])
event_time_extra = int(time_list[-1]) if len(time_list) > 1 else 0

display(player_url,player_id,player_name,event_type,event_score,event_time_label,event_time_minute,event_time_extra)

'/hugo-ekitike/profil/spieler/709726'

'709726'

'Hugo Ekitiké'

'icon-tor-formation'

'1:0'

"37'"

37

0

In [4]:
all_leagues = {
	'premier-league' : 'GB1',
	'bundesliga' : 'L1',
	'serie-a' : 'IT1',
	'laliga' : 'ES1',
	'ligue-1' : 'FR1',
	'campeonato-brasileiro-serie-a' : 'BRA1'
}

In [ ]:
def get_events(headers, league, n_season, n_round):
    """
    Extracts all match events from a specific league round on Transfermarkt.

    The function accesses the Transfermarkt matchday page for the selected
    league, season, and round. It collects the events registered for each
    match, including player information, event type, score at the time of
    the event, and the minute in which the event occurred.

    Parameters:
        headers (dict): HTTP headers used when sending the request.
        league (str): League identifier used in the Transfermarkt URL.
        n_season (int): Starting year of the season.
        n_round (int): Round number to be scraped.

    Returns:
        list: A list of dictionaries where each dictionary represents
        one match event.
    """
    # Build the Transfermarkt URL for the selected league, season, and round
    # Request the page and create a BeautifulSoup object for HTML parsing
    url = f'https://www.transfermarkt.com/{league}/spieltag/wettbewerb/{all_leagues[league]}/saison_id/{n_season}/spieltag/{n_round}'
    response = get_page(url, headers)
    soup = BeautifulSoup(response.content, "lxml")

    # Find the tables containing the matches from the selected round
    all_matches = soup.find_all('table',{'style':'border-top: 0 !important;'})

    # Create a unique identifier for the season
    season_id = f'{all_leagues[league]}-{n_season}'

    # Store all extracted match events
    output_list = []

    # Iterate through every match in the round
    for m, match in enumerate(all_matches):
        # Create a unique identifier for the match
        match_id = f'M-{n_season}-{n_round:02d}-{m+1:02d}'
        # Extract the URL of the match page
        match_url = match.find('td',{'class':'spieltagsansicht-ergebnis'}).find('a').get('href')

        # Find all rows containing events from the current match
        match_event = match.find_all('tr',{'class':'no-border spieltagsansicht-aktionen'})
        # Extract information from each event
        for event in match_event:
            # PLAYER INFORMATION

            # Extract the player's Transfermarkt URL
            player_url = event.find('td',{'class':'spieltagsansicht'}).find('a').get('href')
            # Extract the player ID from the end of the URL
            player_id = int(player_url.split('/')[-1])
            # Extract the player's name from the link title
            player_name = event.find('td',{'class':'spieltagsansicht'}).find('a').get('title')

            # EVENT INFORMATION

            # Identify the event type from the icon's CSS class
            event_type = event.find('span',{'class':'icons_sprite'}).get('class')[-1]
            # Check whether a score is associated with the event
            check = event.find('td',{'class':'zentriert hauptlink'})
            # Store the score when available
            event_score = None if check == None else check.string

            # EVENT TIME INFORMATION

            # Transfermarkt stores event times in different columns
            # depending on whether the event belongs to the home or away team
            home='links'
            away='rechts'

            # Helper function for extracting the time value from either column
            check = lambda x: event.find('td',{'class':f'zentriert no-border-{x}'}).string
            # Select the column containing the actual event time
            event_time_label = check(away) if check(home) == '\xa0' else check(home)

            # Remove the apostrophe and separate regular and stoppage time
            time_list = re.sub("[']",'', event_time_label).split('+')
            # Extract the regular match minute
            event_time_minute = int(time_list[0])
            # Extract stoppage time when available, otherwise default to zero
            event_time_extra = int(time_list[-1]) if len(time_list) > 1 else 0

            # Combine all extracted values into a single event record
            temp = {
                'season_id': season_id,
                'match_id': match_id,
                'match_url': match_url,
                'player_url': player_url,
                'player_id': player_id,
                'player_name': player_name,
                'event_type': event_type,
                'event_score': event_score,
                'event_time_label': event_time_label,
                'event_time_minute': event_time_minute,
                'event_time_extra': event_time_extra
            }

            # Add the event record to the final output
            output_list.append(temp)
    # Return all events extracted from the selected round
    return output_list

In [88]:
ltest = get_events(headers,'premier-league',2025,1)

df = pd.DataFrame(ltest)

display(df)

,season_id,match_id,match_url,player_url,player_id,player_name,event_type,event_score,event_time_label,event_time_minute,event_time_extra
0,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/hugo-ekitike/profil/spieler/709726,709726,Hugo Ekitiké,icon-tor-formation,1:0,37',37,0
1,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/cody-gakpo/profil/spieler/434675,434675,Cody Gakpo,icon-tor-formation,2:0,49',49,0
2,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/antoine-semenyo/profil/spieler/583255,583255,Antoine Semenyo,icon-tor-formation,2:1,64',64,0
3,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/antoine-semenyo/profil/spieler/583255,583255,Antoine Semenyo,icon-tor-formation,2:2,76',76,0
4,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/federico-chiesa/profil/spieler/341092,341092,Federico Chiesa,icon-tor-formation,3:2,88',88,0
5,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/mohamed-salah/profil/spieler/148455,148455,Mohamed Salah,icon-tor-formation,4:2,90+4',90,4
6,GB1-2025,M-2025-01-02,/spielbericht/index/spielbericht/4625775,/ezri-konsa/profil/spieler/413403,413403,Ezri Konsa,icon-rotekarte-formation,None,66',66,0
7,GB1-2025,M-2025-01-03,/spielbericht/index/spielbericht/4625777,/matt-oriley/profil/spieler/406634,406634,Matt O'Riley,icon-elfmeter-formation,1:0,55',55,0
8,GB1-2025,M-2025-01-03,/spielbericht/index/spielbericht/4625777,/rodrigo-muniz/profil/spieler/735571,735571,Rodrigo Muniz,icon-tor-formation,1:1,90+6',90,6
9,GB1-2025,M-2025-01-04,/spielbericht/index/spielbericht/4625778,/eliezer-mayenda/profil/spieler/967346,967346,Eliezer Mayenda,icon-tor-formation,1:0,61',61,0


In [ ]:
def get_matches(headers, league, n_season, n_round):
    """
    Extracts information about all matches from a specific league round
    on Transfermarkt.

    The function accesses the Transfermarkt matchday page for the selected
    league, season, and round. For each match, it collects information about
    the home and away teams, final result, match date, referee, attendance,
    kickoff time, and the corresponding Transfermarkt match URL.

    Parameters:
        headers (dict): HTTP headers used when sending the request.
        league (str): League identifier used in the Transfermarkt URL.
        n_season (int): Starting year of the season.
        n_round (int): Round number to be scraped.

    Returns:
        list: A list of dictionaries where each dictionary contains
        information about one match from the selected round.
    """
    # Build the Transfermarkt URL for the selected league, season, and round
    # Request the page and create a BeautifulSoup object for HTML parsing
    url = f'https://www.transfermarkt.com/{league}/spieltag/wettbewerb/{all_leagues[league]}/saison_id/{n_season}/spieltag/{n_round}'
    response = get_page(url, headers)
    soup = BeautifulSoup(response.content, "lxml")

    # Find all tables containing matches from the selected round
    all_matches = soup.find_all('table',{'style':'border-top: 0 !important;'})

    # CSS classes used to identify the home and away team cells
    home_team = 'hauptlink zentriert no-border-links no-border-rechts hide-for-small spieltagsansicht-wappen'
    away_team = 'hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen'

    # Create a unique identifier for the season
    season_id = f'{all_leagues[league]}-{n_season}'

    # Store the information extracted from each match
    output_list = []

    # Iterate through every match found in the round
    for m, match in enumerate(all_matches):
        # Create a unique identifier for the current match
        match_id = f'M-{n_season}-{n_round:02d}-{m+1:02d}'
        # Extract the URL of the individual match page
        match_url = match.find('td',{'class':'spieltagsansicht-ergebnis'}).find('a').get('href')

        # AWAY TEAM INFORMATION

        # Locate the cell containing the away team information
        away_team_info = match.find_all('td',{'class':away_team})
        # Extract the team's Transfermarkt URL
        away_team_url = away_team_info[0].find('a').get('href')
        # Extract the team ID from the Transfermarkt URL
        away_team_id = int(away_team_url.split('/')[-3])
        # Extract the official team name
        away_team_name = away_team_info[0].find('a').get('title')

        # HOME TEAM INFORMATION

        # Locate the cell containing the home team information
        home_team_info = match.find_all('td',{'class':home_team})
        # Extract the team's Transfermarkt URL
        home_team_url = home_team_info[0].find('a').get('href')
        # Extract the team ID from the Transfermarkt URL
        home_team_id = int(home_team_url.split('/')[-3])
        # Extract the official team name
        home_team_name = home_team_info[0].find('a').get('title')

        # Extract the final score of the match
        match_result = match.find('span',{'class':'matchresult finished'}).string

        # ADDITIONAL MATCH INFORMATION

        # Locate the cells containing date, referee, attendance, and time data
        match_info = match.find_all('td',{'class':'zentriert no-border'})

        # Extract the match date from the URL linked to the date
        match_day = match_info[0].find('a').get('href').split('/')[-1]
        # Extract the referee's name
        match_referee = match_info[1].find('a').string
        # Extract the attendance value as displayed on the page
        match_attendance = match_info[2].get_text().strip()
        # Remove the thousands separator and convert attendance to an integer
        match_attendance = int(re.sub('[.]','', match_attendance).split(' ')[0])
        # Extract the kickoff time text located after the match date link
        # Separate the kickoff time from its AM/PM period
        time_info = match_info[0].find('a').next_sibling.strip().removeprefix('-').strip().split(' ')
        match_time = time_info[0]
        match_time_period = time_info[-1]

        # Combine all extracted values into a single match record
        temp = {
            'season_id': season_id,
            'match_id': match_id,
            'match_url': match_url,
            'home_team_url': home_team_url,
            'home_team_id': home_team_id,
            'home_team_name': home_team_name,
            'match_result': match_result,
            'away_team_url': away_team_url,
            'away_team_id': away_team_id,
            'away_team_name': away_team_name,
            'match_day': match_day,
            'match_referee': match_referee,
            'match_attendance': match_attendance,
            'match_time': match_time,
            'match_time_period': match_time_period
        }

        # Add the current match record to the final output
        output_list.append(temp)
    # Return all matches extracted from the selected round
    return output_list

In [85]:
ltest2 = get_matches(headers,'premier-league',2025,1)

df2 = pd.DataFrame(ltest2)

display(df2)

,season_id,match_id,match_url,home_team_url,home_team_id,home_team_name,match_result,away_team_url,away_team_id,away_team_name,match_day,match_referee,match_attendance,match_time,match_time_period
0,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/fc-liverpool/spielplan/verein/31/saison_id/2025,31,Liverpool FC,4:2,/afc-bournemouth/spielplan/verein/989/saison_i...,989,AFC Bournemouth,2025-08-15,Anthony Taylor,60315,9:00,PM
1,GB1-2025,M-2025-01-02,/spielbericht/index/spielbericht/4625775,/aston-villa/spielplan/verein/405/saison_id/2025,405,Aston Villa,0:0,/newcastle-united/spielplan/verein/762/saison_...,762,Newcastle United,2025-08-16,Craig Pawson,42526,1:30,PM
2,GB1-2025,M-2025-01-03,/spielbericht/index/spielbericht/4625777,/brighton-amp-hove-albion/spielplan/verein/123...,1237,Brighton & Hove Albion,1:1,/fc-fulham/spielplan/verein/931/saison_id/2025,931,Fulham FC,2025-08-16,Samuel Barrott,31478,4:00,PM
3,GB1-2025,M-2025-01-04,/spielbericht/index/spielbericht/4625778,/afc-sunderland/spielplan/verein/289/saison_id...,289,Sunderland AFC,3:0,/west-ham-united/spielplan/verein/379/saison_i...,379,West Ham United,2025-08-16,Robert Jones,46233,4:00,PM
4,GB1-2025,M-2025-01-05,/spielbericht/index/spielbericht/4625779,/tottenham-hotspur/spielplan/verein/148/saison...,148,Tottenham Hotspur,3:0,/fc-burnley/spielplan/verein/1132/saison_id/2025,1132,Burnley FC,2025-08-16,Michael Oliver,61077,4:00,PM
5,GB1-2025,M-2025-01-06,/spielbericht/index/spielbericht/4625780,/wolverhampton-wanderers/spielplan/verein/543/...,543,Wolverhampton Wanderers,0:4,/manchester-city/spielplan/verein/281/saison_i...,281,Manchester City,2025-08-16,Jarred Gillett,31118,6:30,PM
6,GB1-2025,M-2025-01-07,/spielbericht/index/spielbericht/4625776,/nottingham-forest/spielplan/verein/703/saison...,703,Nottingham Forest,3:1,/fc-brentford/spielplan/verein/1148/saison_id/...,1148,Brentford FC,2025-08-17,Peter Bankes,29949,3:00,PM
7,GB1-2025,M-2025-01-08,/spielbericht/index/spielbericht/4625781,/fc-chelsea/spielplan/verein/631/saison_id/2025,631,Chelsea FC,0:0,/crystal-palace/spielplan/verein/873/saison_id...,873,Crystal Palace,2025-08-17,Darren England,39678,3:00,PM
8,GB1-2025,M-2025-01-09,/spielbericht/index/spielbericht/4625782,/manchester-united/spielplan/verein/985/saison...,985,Manchester United,0:1,/fc-arsenal/spielplan/verein/11/saison_id/2025,11,Arsenal FC,2025-08-17,Simon Hooper,73475,5:30,PM
9,GB1-2025,M-2025-01-10,/spielbericht/index/spielbericht/4625783,/leeds-united/spielplan/verein/399/saison_id/2025,399,Leeds United,1:0,/fc-everton/spielplan/verein/29/saison_id/2025,29,Everton FC,2025-08-18,Chris Kavanagh,36820,9:00,PM


In [15]:
url2 = f'https://www.transfermarkt.com/premier-league/torschuetzenliste/wettbewerb/GB1/saison_id/2025/altersklasse/alle/detailpos//page/1'
response = get_page(url2, headers)
soup = BeautifulSoup(response.content, "lxml")

In [ ]:
all_content = soup.find_all('tbody')

content = all_content[1].find_all('tr',{'class':['odd','even']})

i = 5

td_player_info = content[i].find_all('td',{'class':'zentriert'})

leaderboard_pos = td_player_info[0].string
country_name = td_player_info[1].find('img').get('title')
player_age = td_player_info[2].string

if td_player_info[3].string == None:
    team_name = td_player_info[3].find('a').get('title')
    team_url = td_player_info[3].find('a').get('href')
    team_id = team_url.split('/')[-3]
else:
    team_name = td_player_info[3].string
    team_url = None 
    team_id = 0

player_name = td_player_info[4].find('a').get('title')
player_url = td_player_info[4].find('a').get('href')
player_id = player_url.split('/')[-5]
macthes_played = td_player_info[4].string
goals = td_player_info[5].string





display(td_player_info,leaderboard_pos,country_name,player_age,team_name,team_url,team_id,player_name,player_url,player_id,macthes_played,goals)

[<td class="zentriert">6</td>,
 <td class="zentriert"><img alt="England" class="flaggenrahmen" src="https://img.a.transfermarkt.technology/flagge/verysmall/189.png?lm=4711" title="England"/><br/><img alt="Jamaica" class="flaggenrahmen" src="https://img.a.transfermarkt.technology/flagge/verysmall/76.png?lm=4711" title="Jamaica"/></td>,
 <td class="zentriert">26</td>,
 <td class="zentriert"><a href="/nottingham-forest/startseite/verein/703/saison_id/2025" title="Nottingham Forest"><img alt="Nottingham Forest" class="" src="https://img.a.transfermarkt.technology/wappen/verysmall/703.png?lm=4711" title="Nottingham Forest"/></a></td>,
 <td class="zentriert"><a href="/morgan-gibbs-white/leistungsdaten/spieler/429014/saison/2025/wettbewerb/GB1" title="Morgan Gibbs-White">37</a></td>,
 <td class="zentriert hauptlink"><a href="/morgan-gibbs-white/leistungsdaten/spieler/429014/saison/2025/wettbewerb/GB1" title="Morgan Gibbs-White">15</a></td>]

'6'

'England'

'26'

'Nottingham Forest'

...

'Morgan Gibbs-White'

'/morgan-gibbs-white/leistungsdaten/spieler/429014/saison/2025/wettbewerb/GB1'

'429014'

'37'

'15'

In [18]:
pages_info = soup.find_all('div', {'class':'pager'})
last_page_link = pages_info[0].find_all('li',{'class':'tm-pagination__list-item tm-pagination__list-item--icon-last-page'})
last_page_number = last_page_link[0].find('a').get('href').split('/')[-1]

display(last_page_number)

'12'

In [ ]:
def get_top_scorers(headers, league, n_season):
    """
    Extracts the complete top-scorers leaderboard for a specific league
    and season from Transfermarkt.

    The function first accesses the first page of the top-scorers ranking
    to determine how many pages are available. It then iterates through
    every leaderboard page and extracts player, team, nationality,
    ranking, matches played, and goals information.

    Parameters:
        headers (dict): HTTP headers used when sending requests to Transfermarkt.
        league (str): League identifier used in the Transfermarkt URL.
        n_season (int): Starting year of the season to be scraped.

    Returns:
        list: A list of dictionaries where each dictionary contains
        information about one player from the top-scorers leaderboard.
    """
    # Build the URL for the first page of the selected season's top-scorers ranking
    # Request the first page and create a BeautifulSoup object for HTML parsing
    url = f'https://www.transfermarkt.com/{league}/torschuetzenliste/wettbewerb/{all_leagues[league]}/saison_id/{n_season}/altersklasse/alle/detailpos//page/1'
    response = get_page(url, headers)
    soup = BeautifulSoup(response.content, "lxml")

    # Locate the pagination section to determine the total number of pages
    pages_info = soup.find_all('div', {'class':'pager'})
    # Find the link that points to the final leaderboard page
    last_page_link = pages_info[0].find_all('li',{'class':'tm-pagination__list-item tm-pagination__list-item--icon-last-page'})
    # Extract the last page number from the URL
    last_page_number = last_page_link[0].find('a').get('href').split('/')[-1]

    # Store all players extracted from the leaderboard
    output_list = []

    # Iterate through every page of the top-scorers ranking
    for n_page in range(1,int(last_page_number)+1):
        # Build the URL for the current leaderboard page
        # Request and parse the current page
        url = f'https://www.transfermarkt.com/{league}/torschuetzenliste/wettbewerb/{all_leagues[league]}/saison_id/{n_season}/altersklasse/alle/detailpos//page/{n_page}'
        response = get_page(url, headers)
        soup = BeautifulSoup(response.content, "lxml")

        # Locate the table body containing the top-scorers ranking
        all_content = soup.find_all('tbody')
        # Extract all player rows, which alternate between "odd" and "even"
        content = all_content[1].find_all('tr',{'class':['odd','even']})

        # Create a unique identifier for the selected season
        season_id = f'{all_leagues[league]}-{n_season}'

        # Process each player from the current leaderboard page
        for row in content:
            # Locate the centered cells containing most leaderboard information
            td_player_info = row.find_all('td',{'class':'zentriert'})

            # Extract the player's position in the top-scorers leaderboard
            leaderboard_pos = int(td_player_info[0].string)
            # Extract the player's nationality from the flag image
            country_name = td_player_info[1].find('img').get('title')
            # Extract the player's age during the selected season
            player_age = int(td_player_info[2].string)

            # Check how the team information is stored in the HTML.
            # In most cases, the team is inside an <a> tag
            if td_player_info[3].string == None:
                team_name = td_player_info[3].find('a').get('title')
                team_url = td_player_info[3].find('a').get('href')
                team_id = int(team_url.split('/')[-3])
            else:
                # If the team information is stored only as plain text (if the player played for more than one team),
                # keep the displayed name and leave URL and ID unavailable
                team_name = td_player_info[3].string
                team_url = None 
                team_id = 0

            # Extract the player's name from the profile link
            player_name = td_player_info[4].find('a').get('title')
            # Extract the player's Transfermarkt profile URL
            player_url = td_player_info[4].find('a').get('href')
            # Extract the Transfermarkt player ID from the profile URL
            player_id = int(player_url.split('/')[-5])
            # Extract the number of matches played during the season
            matches_played = int(td_player_info[4].string)
            # Extract the total number of goals scored
            goals = int(td_player_info[5].string)

            # Combine all extracted values into a single leaderboard record
            temp = {
                'season_id': season_id,
                'player_url': player_url,
                'player_id': player_id,
                'player_name': player_name,
                'player_age': player_age,
                'country_name': country_name,
                'team_url': team_url,
                'team_id': team_id,
                'team_name': team_name,
                'leaderboard_pos': leaderboard_pos,
                'matches_played': matches_played,
                'goals': goals
            }

            # Add the player record to the final output
            output_list.append(temp)
    # Return the complete top-scorers leaderboard
    return output_list


In [83]:
ltest3 = get_top_scorers(headers,'premier-league',2024)

df3 = pd.DataFrame(ltest3)

display(df3)

,season_id,player_url,player_id,player_name,player_age,country_name,team_url,team_id,team_name,leaderboard_pos,matches_played,goals
0,GB1-2024,/mohamed-salah/leistungsdaten/spieler/148455/s...,148455,Mohamed Salah,32,Egypt,/fc-liverpool/startseite/verein/31/saison_id/2024,31,Liverpool FC,1,38,29
1,GB1-2024,/alexander-isak/leistungsdaten/spieler/349066/...,349066,Alexander Isak,25,Sweden,/newcastle-united/startseite/verein/762/saison...,762,Newcastle United,2,34,23
2,GB1-2024,/erling-haaland/leistungsdaten/spieler/418560/...,418560,Erling Haaland,24,Norway,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,3,31,22
3,GB1-2024,/bryan-mbeumo/leistungsdaten/spieler/413039/sa...,413039,Bryan Mbeumo,25,Cameroon,/fc-brentford/startseite/verein/1148/saison_id...,1148,Brentford FC,4,38,20
4,GB1-2024,/chris-wood/leistungsdaten/spieler/108725/sais...,108725,Chris Wood,33,New Zealand,/nottingham-forest/startseite/verein/703/saiso...,703,Nottingham Forest,5,36,20
...,...,...,...,...,...,...,...,...,...,...,...,...
266,GB1-2024,/ross-stewart/leistungsdaten/spieler/447995/sa...,447995,Ross Stewart,28,Scotland,/fc-southampton/startseite/verein/180/saison_i...,180,Southampton FC,267,12,1
267,GB1-2024,/nico-gonzalez/leistungsdaten/spieler/466805/s...,466805,Nico González,23,Spain,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,268,11,1
268,GB1-2024,/ben-chilwell/leistungsdaten/spieler/316125/sa...,316125,Ben Chilwell,28,England,None,0,for 2 clubs,269,8,1
269,GB1-2024,/ferdi-kadioglu/leistungsdaten/spieler/369316/...,369316,Ferdi Kadıoğlu,25,Türkiye,/brighton-amp-hove-albion/startseite/verein/12...,1237,Brighton & Hove Albion,270,6,1


### SQUAD

In [58]:
#f'https://www.transfermarkt.com/{league}/startseite/wettbewerb/{all_leagues[league]}/plus/?saison_id={n_season}'
url3 = 'https://www.transfermarkt.com/premier-league/startseite/wettbewerb/GB1/plus/?saison_id=2023'
response = get_page(url3, headers)
soup = BeautifulSoup(response.content, "lxml")

In [59]:
all_info = soup.find_all('table',{'class':'items'})
table_info = all_info[0].find_all('tr',{'class':['odd','even']})
display(table_info)

[<tr class="odd">
 <td class="zentriert no-border-rechts"><a href="/manchester-city/startseite/verein/281/saison_id/2023" title="Manchester City"><img alt="Manchester City" class="tiny_wappen" src="https://img.a.transfermarkt.technology/wappen/tiny/281.png?lm=4711" title="Manchester City"/></a></td><td class="hauptlink no-border-links"><a href="/manchester-city/startseite/verein/281/saison_id/2023" title="Manchester City">Manchester City</a> <a href="#"><img alt="English Champion 22/23" class="tabelle-erfolg" src="https://img.a.transfermarkt.technology/erfolge/mini/12.png?lm=4711" title="English Champion 22/23"/></a><a href="#"><img alt="FA Cup Winner 22/23" class="tabelle-erfolg" src="https://img.a.transfermarkt.technology/erfolge/mini/29.png?lm=4711" title="FA Cup Winner 22/23"/></a><a href="#"><img alt="UEFA Champions League winner 22/23" class="tabelle-erfolg" src="https://img.a.transfermarkt.technology/erfolge/mini/4.png?lm=4711" title="UEFA Champions League winner 22/23"/></a></t

In [67]:
team_info = table_info[0].find_all('a')

team_url = team_info[0].get('href')
team_id = int(team_url.split('/')[-3])
team_name = team_info[1].string
team_squad = int(team_info[-2].string)
team_value = team_info[-1].string

abv_index = team_value[-1]
if abv_index == 'n': team_value_int = int(float(team_value.replace('€', '').replace('bn', '')) * 1_000_000_000)
elif abv_index == 'm': team_value_int = int(float(team_value.replace('€', '').replace('m', '')) * 1_000_000)
elif abv_index == 'k': team_value_int = int(float(team_value.replace('€', '').replace('k', '')) * 1_000)
else: team_value_int = int(team_value.replace('€', ''))

add_info = table_info[0].find_all('td',{'class':'zentriert'})
team_avg_age = float(add_info[-2].string)
team_foreigners = int(add_info[-1].string)

# bn,m,k
#team_value_int = 0


display(team_info,add_info,team_url,team_id,team_name,team_squad,team_value,team_value_int,team_avg_age,team_foreigners)

[<a href="/manchester-city/startseite/verein/281/saison_id/2023" title="Manchester City"><img alt="Manchester City" class="tiny_wappen" src="https://img.a.transfermarkt.technology/wappen/tiny/281.png?lm=4711" title="Manchester City"/></a>,
 <a href="/manchester-city/startseite/verein/281/saison_id/2023" title="Manchester City">Manchester City</a>,
 <a href="#"><img alt="English Champion 22/23" class="tabelle-erfolg" src="https://img.a.transfermarkt.technology/erfolge/mini/12.png?lm=4711" title="English Champion 22/23"/></a>,
 <a href="#"><img alt="FA Cup Winner 22/23" class="tabelle-erfolg" src="https://img.a.transfermarkt.technology/erfolge/mini/29.png?lm=4711" title="FA Cup Winner 22/23"/></a>,
 <a href="#"><img alt="UEFA Champions League winner 22/23" class="tabelle-erfolg" src="https://img.a.transfermarkt.technology/erfolge/mini/4.png?lm=4711" title="UEFA Champions League winner 22/23"/></a>,
 <a href="/manchester-city/kader/verein/281/saison_id/2023" title="Manchester City">36</a>

[<td class="zentriert no-border-rechts"><a href="/manchester-city/startseite/verein/281/saison_id/2023" title="Manchester City"><img alt="Manchester City" class="tiny_wappen" src="https://img.a.transfermarkt.technology/wappen/tiny/281.png?lm=4711" title="Manchester City"/></a></td>,
 <td class="zentriert"><a href="/manchester-city/kader/verein/281/saison_id/2023" title="Manchester City">36</a></td>,
 <td class="zentriert">25.7</td>,
 <td class="zentriert">21</td>]

'/manchester-city/startseite/verein/281/saison_id/2023'

281

'Manchester City'

36

'€1.46bn'

1460000000

25.7

21

In [ ]:
def get_squad(headers, league, n_season):
    """
    Extracts squad and market value information for every team in a
    specific league and season from Transfermarkt.

    The function accesses the league overview page and collects information
    about each team, including the team name, Transfermarkt ID, squad size,
    average player age, number of foreign players, and total market value.
    The displayed market value is also converted into a numeric integer
    value to simplify future analysis.

    Parameters:
        headers (dict): HTTP headers used when sending the request.
        league (str): League identifier used in the Transfermarkt URL.
        n_season (int): Starting year of the season to be scraped.

    Returns:
        list: A list of dictionaries where each dictionary contains
        squad and market value information for one team.
    """
    # Build the Transfermarkt league overview URL for the selected season
    # Request the page and create a BeautifulSoup object for HTML parsing
    url = f'https://www.transfermarkt.com/{league}/startseite/wettbewerb/{all_leagues[league]}/plus/?saison_id={n_season}'
    response = get_page(url, headers)
    soup = BeautifulSoup(response.content, "lxml")

    # Create a unique identifier for the selected league and season
    season_id = f'{all_leagues[league]}-{n_season}'

    # Locate all tables with the "items" class on the league overview page
    all_info = soup.find_all('table',{'class':'items'})

    # Store the extracted information for all teams
    output_list = []

    # Extract all team rows from the main league table
    table_info = all_info[0].find_all('tr',{'class':['odd','even']})

    # Process each team in the league
    for row in table_info:

        # TEAM INFORMATION

        # Locate all links contained in the current team row
        team_info = row.find_all('a')

        # Extract the team's Transfermarkt profile URL
        team_url = team_info[0].get('href')
        # Extract the Transfermarkt team ID from the profile URL
        team_id = int(team_url.split('/')[-3])
        # Extract the team name
        team_name = team_info[1].string
        # Extract the number of players registered in the squad
        team_squad = int(team_info[-2].string)
        # Extract the team's total market value as displayed by Transfermarkt
        team_value = team_info[-1].string

        # Identify the abbreviation used in the market value
        # (bn = billion, m = million, k = thousand)
        abv_index = team_value[-1]
        if abv_index == 'n': team_value_int = int(float(team_value.replace('€', '').replace('bn', '')) * 1_000_000_000)
        elif abv_index == 'm': team_value_int = int(float(team_value.replace('€', '').replace('m', '')) * 1_000_000)
        elif abv_index == 'k': team_value_int = int(float(team_value.replace('€', '').replace('k', '')) * 1_000)
        else: team_value_int = int(team_value.replace('€', ''))

        # ADDITIONAL SQUAD INFORMATION

        # Locate the centered cells containing age and foreign-player data
        add_info = row.find_all('td',{'class':'zentriert'})
        # Extract the average age of the squad
        team_avg_age = float(add_info[-2].string)
        # Extract the number of foreign players in the squad
        team_foreigners = int(add_info[-1].string)

        # Combine all extracted values into a single team record
        temp = {
            'season_id': season_id,
            'team_url': team_url,
            'team_id': team_id,
            'team_name': team_name,
            'team_squad': team_squad,
            'team_value': team_value,
            'team_value_int': team_value_int,
            'team_avg_age': team_avg_age,
            'team_foreigners': team_foreigners
        }

        # Add the current team record to the final output
        output_list.append(temp)
    # Return squad information for all teams in the selected season
    return output_list
    

In [78]:
ltest4 = get_squad(headers,'premier-league',2025)

display(ltest4)

[{'season_id': 'GB1-2025',
  'team_url': '/manchester-city/startseite/verein/281/saison_id/2025',
  'team_id': 281,
  'team_name': 'Manchester City',
  'team_squad': 43,
  'team_value': '€1.39bn',
  'team_value_int': 1390000000,
  'team_avg_age': 25.2,
  'team_foreigners': 24},
 {'season_id': 'GB1-2025',
  'team_url': '/fc-arsenal/startseite/verein/11/saison_id/2025',
  'team_id': 11,
  'team_name': 'Arsenal FC',
  'team_squad': 40,
  'team_value': '€1.33bn',
  'team_value_int': 1330000000,
  'team_avg_age': 23.9,
  'team_foreigners': 20},
 {'season_id': 'GB1-2025',
  'team_url': '/fc-chelsea/startseite/verein/631/saison_id/2025',
  'team_id': 631,
  'team_name': 'Chelsea FC',
  'team_squad': 42,
  'team_value': '€1.15bn',
  'team_value_int': 1150000000,
  'team_avg_age': 22.4,
  'team_foreigners': 22},
 {'season_id': 'GB1-2025',
  'team_url': '/fc-liverpool/startseite/verein/31/saison_id/2025',
  'team_id': 31,
  'team_name': 'Liverpool FC',
  'team_squad': 45,
  'team_value': '€980.5

In [77]:
premier_squad = []

# transfermarkt only contains values for team_value from 2004
for season in range(2024,2026):
    squad_data = get_squad(headers,'premier-league',season)
    premier_squad.extend(squad_data)

df_premier_squad = pd.DataFrame(premier_squad)
display(df_premier_squad)

,season_id,team_url,team_id,team_name,team_squad,team_value,team_value_int,team_avg_age,team_foreigners
0,GB1-2024,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,44,€1.36bn,1360000000,25.6,27
1,GB1-2024,/fc-chelsea/startseite/verein/631/saison_id/2024,631,Chelsea FC,58,€1.19bn,1190000000,22.4,32
2,GB1-2024,/fc-arsenal/startseite/verein/11/saison_id/2024,11,Arsenal FC,42,€1.16bn,1160000000,24.5,26
3,GB1-2024,/fc-liverpool/startseite/verein/31/saison_id/2024,31,Liverpool FC,35,€950.95m,950950000,24.7,21
4,GB1-2024,/manchester-united/startseite/verein/985/saiso...,985,Manchester United,49,€825.00m,825000000,24.0,29
5,GB1-2024,/tottenham-hotspur/startseite/verein/148/saiso...,148,Tottenham Hotspur,41,€777.30m,777300000,24.0,23
6,GB1-2024,/aston-villa/startseite/verein/405/saison_id/2024,405,Aston Villa,41,€680.55m,680550000,25.2,24
7,GB1-2024,/newcastle-united/startseite/verein/762/saison...,762,Newcastle United,36,€677.53m,677530000,27.5,15
8,GB1-2024,/brighton-amp-hove-albion/startseite/verein/12...,1237,Brighton & Hove Albion,49,€667.58m,667580000,24.2,33
9,GB1-2024,/crystal-palace/startseite/verein/873/saison_i...,873,Crystal Palace,40,€526.85m,526850000,26.2,18


# TITLES

In [90]:
# f'https://www.transfermarkt.com/{league}/erfolge/wettbewerb/{all_leagues[league]}'

url4 = 'https://www.transfermarkt.com/premier-league/erfolge/wettbewerb/GB1'
response = get_page(url4, headers)
soup = BeautifulSoup(response.content, "lxml")

In [91]:
all_info = soup.find_all('table',{'class':'items'})
table_info = all_info[0].find_all('tr',{'class':['odd','even']})
display(table_info)

[<tr class="odd">
 <td class="zentriert">25/26</td><td class="zentriert no-border-rechts"><a href="/fc-arsenal/startseite/verein/11/saison_id/2025" title="Arsenal FC"><img alt="Arsenal FC" class="tiny_wappen" src="https://img.a.transfermarkt.technology/wappen/tiny/11.png?lm=4711" title="Arsenal FC"/></a></td><td class="hauptlink no-border-links"><a href="/fc-arsenal/startseite/verein/11/saison_id/2025" title="Arsenal FC">Arsenal FC</a></td><td class="rechts"><a href="/mikel-arteta/profil/trainer/47620" id="47620" title="Mikel Arteta">Mikel Arteta</a></td></tr>,
 <tr class="even">
 <td class="zentriert">24/25</td><td class="zentriert no-border-rechts"><a href="/fc-liverpool/startseite/verein/31/saison_id/2024" title="Liverpool FC"><img alt="Liverpool FC" class="tiny_wappen" src="https://img.a.transfermarkt.technology/wappen/tiny/31.png?lm=4711" title="Liverpool FC"/></a></td><td class="hauptlink no-border-links"><a href="/fc-liverpool/startseite/verein/31/saison_id/2024" title="Liverpoo

In [101]:
team_info = table_info[0].find_all('a')

team_url = team_info[0].get('href')
team_id = team_url.split('/')[-3]
team_name = team_info[1].string
manager_url = team_info[2].get('href')
manager_id = team_info[2].get('id')
manager_name = team_info[2].string

add_info = table_info[0].find_all('td',{'class':'zentriert'})

season_name = add_info[0].string

display(team_info,add_info,season_name,team_url,team_id,team_name,manager_url,manager_id,manager_name)

[<a href="/fc-arsenal/startseite/verein/11/saison_id/2025" title="Arsenal FC"><img alt="Arsenal FC" class="tiny_wappen" src="https://img.a.transfermarkt.technology/wappen/tiny/11.png?lm=4711" title="Arsenal FC"/></a>,
 <a href="/fc-arsenal/startseite/verein/11/saison_id/2025" title="Arsenal FC">Arsenal FC</a>,
 <a href="/mikel-arteta/profil/trainer/47620" id="47620" title="Mikel Arteta">Mikel Arteta</a>]

[<td class="zentriert">25/26</td>,
 <td class="zentriert no-border-rechts"><a href="/fc-arsenal/startseite/verein/11/saison_id/2025" title="Arsenal FC"><img alt="Arsenal FC" class="tiny_wappen" src="https://img.a.transfermarkt.technology/wappen/tiny/11.png?lm=4711" title="Arsenal FC"/></a></td>]

'25/26'

'/fc-arsenal/startseite/verein/11/saison_id/2025'

'11'

'Arsenal FC'

'/mikel-arteta/profil/trainer/47620'

'47620'

'Mikel Arteta'

In [ ]:
def get_title(headers, league):
    """
    Extracts the championship history for a specific league from Transfermarkt.

    The function accesses the league's title-history page and collects
    information about each championship season, including the winning team,
    the team's Transfermarkt ID and URL, and the manager responsible for
    the title. The extraction stops after the 1992/93 season.

    Parameters:
        headers (dict): HTTP headers used when sending the request.
        league (str): League identifier used in the Transfermarkt URL.

    Returns:
        list: A list of dictionaries where each dictionary represents
        one championship season and its corresponding winner and manager.
    """
    # Build the Transfermarkt URL containing the league's title history
    # Request the page and create a BeautifulSoup object for HTML parsing
    url = f'https://www.transfermarkt.com/{league}/erfolge/wettbewerb/{all_leagues[league]}'
    response = get_page(url, headers)
    soup = BeautifulSoup(response.content, "lxml")

    # Locate the table containing the championship history
    all_info = soup.find_all('table',{'class':'items'})
    # Extract all season rows from the title-history table
    table_info = all_info[0].find_all('tr',{'class':['odd','even']})

    # Store the extracted championship records
    output_list = []

    # Process each championship season
    for row in table_info:

        # TEAM AND MANAGER INFORMATION

        # Locate all links containing team and manager information
        team_info = row.find_all('a')

        # Extract the winning team's Transfermarkt URL
        team_url = team_info[0].get('href')
        # Extract the team's Transfermarkt ID from its URL
        team_id = team_url.split('/')[-3]
        # Extract the name of the championship-winning team
        team_name = team_info[1].string
        # Extract the manager's Transfermarkt profile URL
        manager_url = team_info[2].get('href')
        # Extract the manager ID stored in the HTML element
        manager_id = team_info[2].get('id')
        # Extract the manager's name
        manager_name = team_info[2].string

        # SEASON INFORMATION

        # Locate the centered table cells containing season information
        add_info = row.find_all('td',{'class':'zentriert'})
        # Extract the displayed season label, such as "23/24"
        season_name = add_info[0].string
        # Extract the season starting year from the team URL
        season = team_url.split('/')[-1]
        # Create a unique identifier combining league and season
        season_id = f'{all_leagues[league]}-{season}'

        # Combine all extracted values into a single title record
        temp = {
            'season_id': season_id,
            'season_name': season_name,
            'team_url': team_url,
            'team_id': team_id,
            'team_name': team_name,
            'manager_url': manager_url,
            'manager_id': manager_id,
            'manager_name': manager_name
        }

        # Add the current championship record to the final output
        output_list.append(temp)

        # Stop after the first Premier League season,
        # excluding records from before the competition was created
        if season_name == '92/93': break

    # Return the championship history from 1992/93 onward
    return output_list


In [109]:
ltest5 = get_title(headers,'premier-league')

df5 = pd.DataFrame(ltest5)

display(df5)

,season_id,season_name,team_url,team_id,team_name,manager_url,manager_id,manager_name
0,GB1-2025,25/26,/fc-arsenal/startseite/verein/11/saison_id/2025,11,Arsenal FC,/mikel-arteta/profil/trainer/47620,47620,Mikel Arteta
1,GB1-2024,24/25,/fc-liverpool/startseite/verein/31/saison_id/2024,31,Liverpool FC,/arne-slot/profil/trainer/34822,34822,Arne Slot
2,GB1-2023,23/24,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
3,GB1-2022,22/23,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
4,GB1-2021,21/22,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
5,GB1-2020,20/21,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
6,GB1-2019,19/20,/fc-liverpool/startseite/verein/31/saison_id/2019,31,Liverpool FC,/jurgen-klopp/profil/trainer/118,118,Jürgen Klopp
7,GB1-2018,18/19,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
8,GB1-2017,17/18,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
9,GB1-2016,16/17,/fc-chelsea/startseite/verein/631/saison_id/2016,631,Chelsea FC,/antonio-conte/profil/trainer/3517,3517,Antonio Conte


In [5]:
# f'https://www.transfermarkt.com/{league}/spieltagtabelle/wettbewerb/{all_leagues[league]}/saison_id/{n_season}/spieltag/{n_round}'

url5 = f'https://www.transfermarkt.com/premier-league/spieltagtabelle/wettbewerb/GB1/saison_id/2025/spieltag/1'
response = get_page(url5, headers)
soup = BeautifulSoup(response.content, "lxml")

In [17]:
all_info = soup.find_all('table',{'class':'items'})
# table_info = all_info[0].find_all('td',{'class':['no-border-links hauptlink','zentriert']})

display(all_info)

[<table class="items">
 <thead>
 <tr>
 <th class="rechts" id="yw1_c0">#</th><th colspan="2" id="yw1_c1">Club</th><th class="zentriert" id="yw1_c2"><span class="icons_sprite icon-einsaetze-table-header" title="Matches"> </span></th><th class="zentriert" id="yw1_c3">W</th><th class="zentriert" id="yw1_c4">D</th><th class="zentriert" id="yw1_c5">L</th><th class="zentriert" id="yw1_c6">Goals</th><th class="zentriert" id="yw1_c7">+/-</th><th class="zentriert" id="yw1_c8">Pts</th></tr>
 </thead>
 <tbody>
 <tr>
 <td class="rechts hauptlink" style="background-color: #afd179">1 </td><td class="zentriert no-border-rechts"><a href="/manchester-city/spielplan/verein/281/saison_id/2025" title="Manchester City"><img alt="Manchester City" class="tiny_wappen" src="https://img.a.transfermarkt.technology/wappen/tiny/281.png?lm=4711" title="Manchester City"/></a></td>
 <td class="no-border-links hauptlink">
 <a href="/manchester-city/spielplan/verein/281/saison_id/2025" title="Manchester City">Man City</

In [27]:
table_info = all_info[0].find_all('tr')


team_info = table_info[1].find('td',{'class':'no-border-links hauptlink'})

team_url = team_info.find('a').get('href')
team_id = team_url.split('/')[-3]
team_name = team_info.find('a').get('title')

stats_info = table_info[1].find_all('td',{'class':'zentriert'})

matches_played = stats_info[1].string
matches_won = stats_info[2].string
matches_drawn = stats_info[3].string
matches_losses = stats_info[4].string
goals = stats_info[5].string
goals_dif = stats_info[6].string
points = stats_info[7].string

display(table_info[1],team_info,team_url,team_id,team_name,stats_info,matches_played,matches_won,matches_drawn,matches_losses,goals,goals_dif,points)

<tr>
<td class="rechts hauptlink" style="background-color: #afd179">1 </td><td class="zentriert no-border-rechts"><a href="/manchester-city/spielplan/verein/281/saison_id/2025" title="Manchester City"><img alt="Manchester City" class="tiny_wappen" src="https://img.a.transfermarkt.technology/wappen/tiny/281.png?lm=4711" title="Manchester City"/></a></td>
<td class="no-border-links hauptlink">
<a href="/manchester-city/spielplan/verein/281/saison_id/2025" title="Manchester City">Man City</a>
</td><td class="zentriert">1</td><td class="zentriert">1</td><td class="zentriert">0</td><td class="zentriert">0</td><td class="zentriert">4:0</td><td class="zentriert">4</td><td class="zentriert">3</td></tr>

<td class="no-border-links hauptlink">
<a href="/manchester-city/spielplan/verein/281/saison_id/2025" title="Manchester City">Man City</a>
</td>

'/manchester-city/spielplan/verein/281/saison_id/2025'

'281'

'Manchester City'

...

'0'

'0'

'4:0'

'4'

'3'

In [43]:
def get_placements(headers, league, n_season, n_round):
    """
    Extracts the league table standings for a specific round and season
    from Transfermarkt.

    The function accesses the standings page for the selected league,
    season, and round. For each team, it collects its current position,
    team information, matches played, wins, draws, losses, goals,
    goal difference, and total points.

    Parameters:
        headers (dict): HTTP headers used when sending the request.
        league (str): League identifier used in the Transfermarkt URL.
        n_season (int): Starting year of the season to be scraped.
        n_round (int): Round number used to retrieve the standings.

    Returns:
        list: A list of dictionaries where each dictionary contains
        the league-table information for one team after the selected round.
    """
    # Build the Transfermarkt standings URL for the selected league, season, and round
    # Request the page and create a BeautifulSoup object for HTML parsing
    url = f'https://www.transfermarkt.com/{league}/spieltagtabelle/wettbewerb/{all_leagues[league]}/saison_id/{n_season}/spieltag/{n_round}'
    response = get_page(url, headers)
    soup = BeautifulSoup(response.content, "lxml")

    # Locate the tables containing the league standings
    # Extract all rows from the main standings table
    all_info = soup.find_all('table',{'class':'items'})
    table_info = all_info[0].find_all('tr')

    # Create a unique identifier for the selected league and season
    season_id = f'{all_leagues[league]}-{n_season}'

    # Store the standings information for all teams
    output_list = []

    # Skip the first row because it contains the table headers,
    # then process each team according to its current table position
    for i, row in enumerate(table_info[1:]):
        # TEAM INFORMATION

        # Locate the cell containing the team's name and profile link
        team_info = row.find('td',{'class':'no-border-links hauptlink'})

        # Extract the team's Transfermarkt profile URL
        team_url = team_info.find('a').get('href')
        # Extract the Transfermarkt team ID from the profile URL
        team_id = int(team_url.split('/')[-3])
        # Extract the team name from the link title
        team_name = team_info.find('a').get('title')

        # STANDINGS INFORMATION

        # Locate the cells containing the team's statistical information
        stats_info = row.find_all('td',{'class':'zentriert'})

        # Extract the statistics in the same order in which they appear
        # in the standings table. The first centered cell is excluded
        # because it does not belong to these performance statistics.
        (
            matches_played,
            matches_won,
            matches_draw,
            matches_losses,
            matches_goals,
            goals_dif,
            points
        ) = [stat.string for stat in stats_info[1:8]]

        # Combine the team information and statistics into a single record
        temp = {
            'season_id': season_id,
            'team_url': team_url,
            'team_id': team_id,
            'team_name': team_name,
            'position': i+1,
            'matches_played': int(matches_played),
            'matches_won': int(matches_won),
            'matches_draw': int(matches_draw),
            'matches_losses': int(matches_losses),
            'matches_goals': matches_goals,
            'goals_dif': int(goals_dif),
            'points': int(points)
        }

        # Add the current team's standings record to the final output
        output_list.append(temp)
    # Return the complete league table for the selected round
    return output_list

In [44]:
ltest6 = get_placements(headers,'premier-league',2025,6)

df6 = pd.DataFrame(ltest6)

display(df6)

,season_id,team_url,team_id,team_name,position,matches_played,matches_won,matches_draw,matches_losses,matches_goals,goals_dif,points
0,GB1-2025,/fc-liverpool/spielplan/verein/31/saison_id/2025,31,Liverpool FC,1,6,5,0,1,12:7,5,15
1,GB1-2025,/fc-arsenal/spielplan/verein/11/saison_id/2025,11,Arsenal FC,2,6,4,1,1,12:3,9,13
2,GB1-2025,/crystal-palace/spielplan/verein/873/saison_id...,873,Crystal Palace,3,6,3,3,0,8:3,5,12
3,GB1-2025,/tottenham-hotspur/spielplan/verein/148/saison...,148,Tottenham Hotspur,4,6,3,2,1,11:4,7,11
4,GB1-2025,/afc-sunderland/spielplan/verein/289/saison_id...,289,Sunderland AFC,5,6,3,2,1,7:4,3,11
5,GB1-2025,/afc-bournemouth/spielplan/verein/989/saison_i...,989,AFC Bournemouth,6,6,3,2,1,8:7,1,11
6,GB1-2025,/manchester-city/spielplan/verein/281/saison_i...,281,Manchester City,7,6,3,1,2,14:6,8,10
7,GB1-2025,/fc-chelsea/spielplan/verein/631/saison_id/2025,631,Chelsea FC,8,6,2,2,2,11:8,3,8
8,GB1-2025,/fc-everton/spielplan/verein/29/saison_id/2025,29,Everton FC,9,6,2,2,2,7:6,1,8
9,GB1-2025,/brighton-amp-hove-albion/spielplan/verein/123...,1237,Brighton & Hove Albion,10,6,2,2,2,9:9,0,8


In [49]:
pytest1 = sf.get_events(headers,'premier-league',2025,1)

dfpy1 = pd.DataFrame(pytest1)

display(dfpy1)

,season_id,match_id,match_url,player_url,player_id,player_name,event_type,event_score,event_time_label,event_time_minute,event_time_extra
0,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/hugo-ekitike/profil/spieler/709726,709726,Hugo Ekitiké,icon-tor-formation,1:0,37',37,0
1,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/cody-gakpo/profil/spieler/434675,434675,Cody Gakpo,icon-tor-formation,2:0,49',49,0
2,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/antoine-semenyo/profil/spieler/583255,583255,Antoine Semenyo,icon-tor-formation,2:1,64',64,0
3,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/antoine-semenyo/profil/spieler/583255,583255,Antoine Semenyo,icon-tor-formation,2:2,76',76,0
4,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/federico-chiesa/profil/spieler/341092,341092,Federico Chiesa,icon-tor-formation,3:2,88',88,0
5,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/mohamed-salah/profil/spieler/148455,148455,Mohamed Salah,icon-tor-formation,4:2,90+4',90,4
6,GB1-2025,M-2025-01-02,/spielbericht/index/spielbericht/4625775,/ezri-konsa/profil/spieler/413403,413403,Ezri Konsa,icon-rotekarte-formation,None,66',66,0
7,GB1-2025,M-2025-01-03,/spielbericht/index/spielbericht/4625777,/matt-oriley/profil/spieler/406634,406634,Matt O'Riley,icon-elfmeter-formation,1:0,55',55,0
8,GB1-2025,M-2025-01-03,/spielbericht/index/spielbericht/4625777,/rodrigo-muniz/profil/spieler/735571,735571,Rodrigo Muniz,icon-tor-formation,1:1,90+6',90,6
9,GB1-2025,M-2025-01-04,/spielbericht/index/spielbericht/4625778,/eliezer-mayenda/profil/spieler/967346,967346,Eliezer Mayenda,icon-tor-formation,1:0,61',61,0


In [51]:
pytest2 = sf.get_matches(headers,'premier-league',2025,1)

dfpy2 = pd.DataFrame(pytest2)

display(dfpy2)

,season_id,match_id,match_url,home_team_url,home_team_id,home_team_name,match_result,away_team_url,away_team_id,away_team_name,match_day,match_referee,match_attendance,match_time,match_time_period
0,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/fc-liverpool/spielplan/verein/31/saison_id/2025,31,Liverpool FC,4:2,/afc-bournemouth/spielplan/verein/989/saison_i...,989,AFC Bournemouth,2025-08-15,Anthony Taylor,60315,9:00,PM
1,GB1-2025,M-2025-01-02,/spielbericht/index/spielbericht/4625775,/aston-villa/spielplan/verein/405/saison_id/2025,405,Aston Villa,0:0,/newcastle-united/spielplan/verein/762/saison_...,762,Newcastle United,2025-08-16,Craig Pawson,42526,1:30,PM
2,GB1-2025,M-2025-01-03,/spielbericht/index/spielbericht/4625777,/brighton-amp-hove-albion/spielplan/verein/123...,1237,Brighton & Hove Albion,1:1,/fc-fulham/spielplan/verein/931/saison_id/2025,931,Fulham FC,2025-08-16,Samuel Barrott,31478,4:00,PM
3,GB1-2025,M-2025-01-04,/spielbericht/index/spielbericht/4625778,/afc-sunderland/spielplan/verein/289/saison_id...,289,Sunderland AFC,3:0,/west-ham-united/spielplan/verein/379/saison_i...,379,West Ham United,2025-08-16,Robert Jones,46233,4:00,PM
4,GB1-2025,M-2025-01-05,/spielbericht/index/spielbericht/4625779,/tottenham-hotspur/spielplan/verein/148/saison...,148,Tottenham Hotspur,3:0,/fc-burnley/spielplan/verein/1132/saison_id/2025,1132,Burnley FC,2025-08-16,Michael Oliver,61077,4:00,PM
5,GB1-2025,M-2025-01-06,/spielbericht/index/spielbericht/4625780,/wolverhampton-wanderers/spielplan/verein/543/...,543,Wolverhampton Wanderers,0:4,/manchester-city/spielplan/verein/281/saison_i...,281,Manchester City,2025-08-16,Jarred Gillett,31118,6:30,PM
6,GB1-2025,M-2025-01-07,/spielbericht/index/spielbericht/4625776,/nottingham-forest/spielplan/verein/703/saison...,703,Nottingham Forest,3:1,/fc-brentford/spielplan/verein/1148/saison_id/...,1148,Brentford FC,2025-08-17,Peter Bankes,29949,3:00,PM
7,GB1-2025,M-2025-01-08,/spielbericht/index/spielbericht/4625781,/fc-chelsea/spielplan/verein/631/saison_id/2025,631,Chelsea FC,0:0,/crystal-palace/spielplan/verein/873/saison_id...,873,Crystal Palace,2025-08-17,Darren England,39678,3:00,PM
8,GB1-2025,M-2025-01-09,/spielbericht/index/spielbericht/4625782,/manchester-united/spielplan/verein/985/saison...,985,Manchester United,0:1,/fc-arsenal/spielplan/verein/11/saison_id/2025,11,Arsenal FC,2025-08-17,Simon Hooper,73475,5:30,PM
9,GB1-2025,M-2025-01-10,/spielbericht/index/spielbericht/4625783,/leeds-united/spielplan/verein/399/saison_id/2025,399,Leeds United,1:0,/fc-everton/spielplan/verein/29/saison_id/2025,29,Everton FC,2025-08-18,Chris Kavanagh,36820,9:00,PM


In [54]:
pytest3 = sf.get_top_scorers(headers,'premier-league',2025)

dfpy3 = pd.DataFrame(pytest3)

display(dfpy3)

,season_id,player_url,player_id,player_name,player_age,country_name,team_url,team_id,team_name,leaderboard_pos,matches_played,goals
0,GB1-2025,/erling-haaland/leistungsdaten/spieler/418560/...,418560,Erling Haaland,25,Norway,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,1,35,27
1,GB1-2025,/igor-thiago/leistungsdaten/spieler/739443/sai...,739443,Igor Thiago,24,Brazil,/fc-brentford/startseite/verein/1148/saison_id...,1148,Brentford FC,2,38,22
2,GB1-2025,/antoine-semenyo/leistungsdaten/spieler/583255...,583255,Antoine Semenyo,26,Ghana,None,0,for 2 clubs,3,37,17
3,GB1-2025,/ollie-watkins/leistungsdaten/spieler/324358/s...,324358,Ollie Watkins,30,England,/aston-villa/startseite/verein/405/saison_id/2025,405,Aston Villa,4,37,16
4,GB1-2025,/joao-pedro/leistungsdaten/spieler/626724/sais...,626724,João Pedro,24,Brazil,/fc-chelsea/startseite/verein/631/saison_id/2025,631,Chelsea FC,5,35,15
...,...,...,...,...,...,...,...,...,...,...,...,...
275,GB1-2025,/matt-oriley/leistungsdaten/spieler/406634/sai...,406634,Matt O'Riley,25,Denmark,/brighton-amp-hove-albion/startseite/verein/12...,1237,Brighton & Hove Albion,276,6,1
276,GB1-2025,/fabio-carvalho/leistungsdaten/spieler/559263/...,559263,Fábio Carvalho,23,Portugal,/fc-brentford/startseite/verein/1148/saison_id...,1148,Brentford FC,277,6,1
277,GB1-2025,/lorenzo-lucca/leistungsdaten/spieler/572265/s...,572265,Lorenzo Lucca,25,Italy,/nottingham-forest/startseite/verein/703/saiso...,703,Nottingham Forest,278,4,1
278,GB1-2025,/ben-davies/leistungsdaten/spieler/192765/sais...,192765,Ben Davies,32,Wales,/tottenham-hotspur/startseite/verein/148/saiso...,148,Tottenham Hotspur,279,3,1


In [57]:
pytest4 = sf.get_squad(headers,'premier-league',2025)

dfpy4 = pd.DataFrame(pytest4)

display(dfpy4)

,season_id,team_url,team_id,team_name,team_squad,team_value,team_value_int,team_avg_age,team_foreigners
0,GB1-2025,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,43,€1.39bn,1390000000,25.2,24
1,GB1-2025,/fc-arsenal/startseite/verein/11/saison_id/2025,11,Arsenal FC,40,€1.33bn,1330000000,23.9,20
2,GB1-2025,/fc-chelsea/startseite/verein/631/saison_id/2025,631,Chelsea FC,42,€1.15bn,1150000000,22.4,22
3,GB1-2025,/fc-liverpool/startseite/verein/31/saison_id/2025,31,Liverpool FC,45,€980.50m,980500000,24.0,29
4,GB1-2025,/tottenham-hotspur/startseite/verein/148/saiso...,148,Tottenham Hotspur,45,€813.40m,813400000,24.0,28
5,GB1-2025,/manchester-united/startseite/verein/985/saiso...,985,Manchester United,34,€774.60m,774600000,24.9,22
6,GB1-2025,/crystal-palace/startseite/verein/873/saison_i...,873,Crystal Palace,41,€702.50m,702500000,25.2,22
7,GB1-2025,/newcastle-united/startseite/verein/762/saison...,762,Newcastle United,37,€697.70m,697700000,26.6,14
8,GB1-2025,/afc-bournemouth/startseite/verein/989/saison_...,989,AFC Bournemouth,39,€680.98m,680980000,25.0,29
9,GB1-2025,/nottingham-forest/startseite/verein/703/saiso...,703,Nottingham Forest,42,€638.68m,638680000,25.5,30


In [61]:
pytest5 = sf.get_title(headers,'premier-league')

dfpy5 = pd.DataFrame(pytest5)

display(dfpy5)

,season_id,season_name,team_url,team_id,team_name,manager_url,manager_id,manager_name
0,GB1-2025,25/26,/fc-arsenal/startseite/verein/11/saison_id/2025,11,Arsenal FC,/mikel-arteta/profil/trainer/47620,47620,Mikel Arteta
1,GB1-2024,24/25,/fc-liverpool/startseite/verein/31/saison_id/2024,31,Liverpool FC,/arne-slot/profil/trainer/34822,34822,Arne Slot
2,GB1-2023,23/24,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
3,GB1-2022,22/23,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
4,GB1-2021,21/22,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
5,GB1-2020,20/21,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
6,GB1-2019,19/20,/fc-liverpool/startseite/verein/31/saison_id/2019,31,Liverpool FC,/jurgen-klopp/profil/trainer/118,118,Jürgen Klopp
7,GB1-2018,18/19,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
8,GB1-2017,17/18,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,/pep-guardiola/profil/trainer/5672,5672,Pep Guardiola
9,GB1-2016,16/17,/fc-chelsea/startseite/verein/631/saison_id/2016,631,Chelsea FC,/antonio-conte/profil/trainer/3517,3517,Antonio Conte


In [65]:
pytest6 = sf.get_placements(headers,'premier-league',2025,1)

dfpy6 = pd.DataFrame(pytest6)

display(dfpy6)

,season_id,team_url,team_id,team_name,position,matches_played,matches_won,matches_draw,matches_losses,matches_goals,goals_dif,points
0,GB1-2025,/manchester-city/spielplan/verein/281/saison_i...,281,Manchester City,1,1,1,0,0,4:0,4,3
1,GB1-2025,/afc-sunderland/spielplan/verein/289/saison_id...,289,Sunderland AFC,2,1,1,0,0,3:0,3,3
2,GB1-2025,/tottenham-hotspur/spielplan/verein/148/saison...,148,Tottenham Hotspur,3,1,1,0,0,3:0,3,3
3,GB1-2025,/fc-liverpool/spielplan/verein/31/saison_id/2025,31,Liverpool FC,4,1,1,0,0,4:2,2,3
4,GB1-2025,/nottingham-forest/spielplan/verein/703/saison...,703,Nottingham Forest,5,1,1,0,0,3:1,2,3
5,GB1-2025,/fc-arsenal/spielplan/verein/11/saison_id/2025,11,Arsenal FC,6,1,1,0,0,1:0,1,3
6,GB1-2025,/leeds-united/spielplan/verein/399/saison_id/2025,399,Leeds United,7,1,1,0,0,1:0,1,3
7,GB1-2025,/brighton-amp-hove-albion/spielplan/verein/123...,1237,Brighton & Hove Albion,8,1,0,1,0,1:1,0,1
8,GB1-2025,/fc-fulham/spielplan/verein/931/saison_id/2025,931,Fulham FC,9,1,0,1,0,1:1,0,1
9,GB1-2025,/aston-villa/spielplan/verein/405/saison_id/2025,405,Aston Villa,10,1,0,1,0,0:0,0,1
